# 12 · 计划与持续任务：谁允许下一轮开始

计划展示进度；Goal 记录持续意图与预算；Scheduler 决定是否实际启动。把三者混在一起，会让“还有一个待办”变成未经授权的续跑，
或者让刚到达的用户输入与自动下一轮争抢执行权。我们用不可变快照复现资格判定、累计用量记账和启动前重查。
本章不创建应用里的真实 Goal，也没有 Scheduler 进程：`record_started` 只记宿主已经启动的回执。

**运行范围**：本章直接使用最终包中的共享组件，逐格验证本专题机制；不会启动模型、访问真实收费 API 或使用个人凭据。每次从第一格顺序运行，运行文件进入独立的 `runs/` 目录。改动参数后，从相应状态初始化格重新运行。


In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'code/mini_codex/__init__.py').is_file())
sys.path.insert(0, str(ROOT/'code'))
from mini_codex.learning import load_example, show_source, fresh_run_dir, run_python, run_chapter
from unittest import TestCase
expect = TestCase()  # 只借用异常断言；下面逐格调用实现，不运行总测试替代实验。


## 源码事实与教学选择

固定对照提交：`c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee`。以下链接说明 Codex 对应机制；Python 类型和实验状态机是教学选择，运行它们不等于运行 Codex。

- [S01 · PlanHandler::handle_call](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/src/tools/handlers/plan.rs#L65-L111)：PlanHandler在非Plan模式解析参数后只发布PlanUpdate并返回Plan updated；步骤状态是pending/in_progress/completed。
- [S06 · GoalRuntimeHandle::continue_if_idle](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/ext/goal/src/runtime.rs#L434-L530)：Goal idle read/start与外部set/clear prepare/write使用同一goal_state_permit；只有Active且无deferral才向start_turn_if_idle提交，Started后标记自动turn。
- [S09 · record_token_usage; record_descendant_token_usage](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/ext/goal/src/accounting.rs#L282-L310)：Goal增量计量为累计usage求差后(input-cached_input)+max(output,0)；后代last usage经独立累计量汇入根目标。


## 实验 1 · 修改计划并不创建 Goal


先更新计划并拒绝两个同时进行的步骤。这个数量检查是教学版加严；固定源码工具描述也要求最多一个，
但不能据此宣称 PlanHandler 实现了同样的循环校验。创建 Goal 需要宿主提供的授权事实。


In [2]:
g = load_example('12', 'goals')
from dataclasses import replace
plan = g.update_plan(g.Plan(), [('找到入口','completed'), ('验证筛选','in_progress')])
assert plan.revision == 1
assert g.qualify_continue(None, g.IdleFacts('idle-1', 1)).reason == 'no_goal'
with expect.assertRaisesRegex(ValueError, 'multiple_in_progress'):
    g.update_plan(plan, [('A','in_progress'), ('B','in_progress')])
with expect.assertRaisesRegex(ValueError, 'goal_not_authorized'):
    g.set_goal('g1', 'Finish task-list verification', authority='model', limit=100)
goal = g.set_goal('g1', 'Finish task-list verification', authority='user', limit=100)
print({'plan_revision': plan.revision, 'goal_status': goal.status})


{'plan_revision': 1, 'goal_status': 'active'}


## 实验 2 · 累计计量必须先求差


同一 producer 上报的是累计值。重放不能重复扣费，旧序号不能倒退水位，序号相同但内容不同必须拒绝。
教学计量使用 `(input − cached) + output`；它是预算单位，不是现实账单价格。改变 `LIMIT` 观察达到边界后拒绝新轮。


In [3]:
LIMIT = 50
budget_goal = g.set_goal('budget', 'Bounded experiment', authority='user', limit=LIMIT)
f1 = g.Usage('budget', 'turn-1/request-1', 1, 30, 10, 5)
f2 = g.Usage('budget', 'turn-1/request-1', 2, 50, 20, 15)
budget_goal = g.account(budget_goal, f1)
assert budget_goal.used == 25
assert g.account(budget_goal, f1) == budget_goal
budget_goal = g.account(budget_goal, f2)
assert budget_goal.used == 45
assert g.account(budget_goal, f1) == budget_goal
with expect.assertRaisesRegex(ValueError, 'conflicting_duplicate'):
    g.account(budget_goal, replace(f2, output=16))
budget_goal = g.account(budget_goal, g.Usage('budget','turn-2/request-1',1,10,0,0))
assert budget_goal.used == 55
assert (budget_goal.status == 'budget_limited') == (55 >= LIMIT)
print({'used': budget_goal.used, 'limit': LIMIT, 'state': budget_goal.status})


{'used': 55, 'limit': 50, 'state': 'budget_limited'}


## 实验 3 · 等到空闲还不够，启动前再次验证


候选记录目标 revision、空闲事件和调度代次。在候选产生后插入用户输入、暂停目标或推进 epoch，
都应使旧候选失效。真正集成时重查与 StartIfIdle 必须在同一互斥准入区内；本纯函数不提供那把锁。


In [4]:
facts = g.IdleFacts('idle-1', 3)
decision = g.qualify_continue(goal, facts)
assert decision.eligible and g.revalidate(goal, decision.candidate, facts)
assert not g.revalidate(goal, decision.candidate, replace(facts, pending_user_input=True))
assert not g.revalidate(goal, decision.candidate, replace(facts, scheduler_epoch=4))
paused = g.pause(goal, explicit_user_request=True)
assert not g.revalidate(paused, decision.candidate, facts)
# 合成宿主前提：StartIfIdle 已返回 Started，才写入这个回执。
receipted = g.record_started(goal, decision.candidate)
assert g.qualify_continue(receipted, facts).reason == 'duplicate_idle_event'
print({'candidate': decision.candidate, 'receipt_revision': receipted.revision})


{'candidate': Candidate(goal_id='g1', revision=1, event_id='idle-1', scheduler_epoch=3), 'receipt_revision': 2}


## 实验 4 · 取消不自动成为暂停，恢复先确认状态


暂停是明确用户控制；恢复要求宿主已查清在途动作、预算和调度状态。把未知默认为空闲会重复副作用。
逐个关闭准入事实，观察拒绝原因。这里重建快照只是状态判定实验，不是持久化崩溃恢复。


In [5]:
with expect.assertRaisesRegex(ValueError, 'pause_not_authorized'):
    g.pause(goal, explicit_user_request=False)
blocked_cases = {
    'recovery_confirmed': 'recovery_unconfirmed',
    'budget_known': 'budget_unknown',
    'idle': 'not_idle',
}
for field, expected in blocked_cases.items():
    result = g.qualify_continue(goal, replace(facts, **{field: False}))
    assert not result.eligible and result.reason == expected
    print(field, '→', result.reason)
recovered = replace(facts, recovery_confirmed=True, budget_known=True)
assert g.qualify_continue(goal, recovered).eligible


recovery_confirmed → recovery_unconfirmed
budget_known → budget_unknown
idle → not_idle


## 实验 5 · 停止条件也需要证据


连续空自动轮达到三次时停止，重复结束事件不计两次。完成必须有宿主验证引用；字符串存在只体现这个局部接口约束，
不证明引用确实可信。真实集成需要在宿主解析验证记录，而非让模型自由填写。


In [6]:
empty = goal
for turn in ('auto-1','auto-2','auto-3'):
    empty = g.turn_ended(empty, turn, automatic=True, had_activity=False, empty_final=True)
assert empty.status == 'blocked' and empty.empty_turns == 3
assert g.turn_ended(empty, 'auto-3', automatic=True, had_activity=False, empty_final=True) == empty
with expect.assertRaisesRegex(ValueError, 'completion_not_verified'):
    g.complete(goal, verified_snapshot=None)
completed = g.complete(goal, verified_snapshot='synthetic-verifier:snapshot-1')
assert completed.status == 'complete'
assert not g.qualify_continue(completed, facts).eligible
print({'empty_turn_stop': empty.reason, 'verified_stop': completed.reason})


{'empty_turn_stop': 'three_empty_automatic_turns', 'verified_stop': 'verified_completion'}


## 你现在可以解释什么

能区分进度、持续意图与实际执行权，能指出任何 await 或锁释放之后哪些资格可能过期。这里没有真正启动下一轮，没有持久 journal，也未验证跨进程并发。

**验证边界**：本章断言验证当前局部实现与合成输入。未运行固定提交的 Rust 测试，未验证真实模型、外部服务或累计 Agent 的端到端行为。
